In [1]:
import pandas as pd
import numpy as np

def read_excel(file_name):
    df = pd.read_excel(file_name)
    return df

def read_txt(file_name):
    file = open(file_name)
    lines = file.readlines()
    return(lines[0])

In [2]:
import os
import glob

def get_files(subfolder, extension):
    dir = f"{os.getcwd()}/content/{subfolder}/"
    tables = glob.glob(f"{dir}*.{extension}")
    return tables

In [3]:
class Analizer:
    def __init__(self, boundary):
        self.results = get_files(subfolder="results", extension="xlsx")
        self.results_df = pd.DataFrame()
        self.boundary = boundary
    
    def has_minimum_requirements(self, df, sort_by="r2"):
        sorted_df = df.sort_values(by=sort_by, ascending=False)
        top_r2 = sorted_df.head(1)[sort_by].values[0]
        if top_r2 < self.boundary:  # aqui mantemos menor que o limite
            return True
        return False
    
    def concatenate_df(self, df, architecture):
        if self.has_minimum_requirements(df):
            df['Architecture'] = architecture
            df = df.rename(columns={'Unnamed: 0': 'model'})
            self.results_df = pd.concat([self.results_df, df], ignore_index=True) 

    def create_results_df(self):
        for file in self.results:
            df = read_excel(file)
            architecture = read_txt(file.replace(".xlsx", ".txt"))
            self.concatenate_df(df, architecture)
        self.results_df = self.results_df.sort_values(by="r2", ascending=False, ignore_index=True)

    def discard_below_average(self, sort_by):
        column_mean = self.results_df[sort_by].mean()      
        self.results_df = self.results_df[self.results_df[sort_by] >= column_mean]
    
    def discard_high_standard_deviation(self):
        r2_val, r2_test = self.results_df['r2_val'], self.results_df['r2_test']
        std_devs = np.abs(r2_val - r2_test)
        mean_std_dev = std_devs.mean()
        self.results_df = self.results_df[std_devs < mean_std_dev]

    def clean_folder(self, subfolder, extension, remove_last=True):
        files = get_files(subfolder, extension)
        models = self.results_df["model"]
        if (remove_last):
            models = models.apply(lambda x: '_'.join(x.rsplit('_', 1)[:-1]))
        for file in files:
            file_name = os.path.basename(file).split('.')[0]
            file_parts = file_name.split('_')            
            dataset_model = f"model_{file_parts[1]}_{file_parts[2]}" 
            if (remove_last == False):
                dataset_model = (f"{dataset_model}_{file_parts[3]}")
            if dataset_model not in models.values:
                os.remove(file)   
        
    def Analize(self):
        self.create_results_df()
        self.discard_below_average(sort_by="r2")
        self.discard_below_average(sort_by="r2_vt")
        self.discard_high_standard_deviation()
        self.results_df.to_excel(f"better_results.xlsx", index=True)
        display(self.results_df)


In [4]:
analize = Analizer(0.8)
analize.Analize()
analize.clean_folder(subfolder="dataset", extension="pkl")
analize.clean_folder(subfolder="results", extension="xlsx")
analize.clean_folder(subfolder="results", extension="txt")
analize.clean_folder(subfolder="models", extension="keras", remove_last=False)



,model,r2,r2_sup,r2_test,r2_val,r2_vt,mse,mse_sup,mse_test,mse_val,mse_vt,mape,rmse,r2_adj,rsd,aic,bic,Architecture
1,model_24_0_0,0.231033,0.197164,-0.417872,-11.465793,-1.696789,1.259191,1.314652,0.289982,0.579652,0.434817,1.360697,1.122137,1.323776,1.169908,161.539062,260.268003,"Hidden Size=[20], regularizer=0.05, learning_r..."
2,model_4_0_0,0.200002,0.062761,-1.367765,-0.106357,-0.559406,1.310004,1.534737,0.406476,0.334564,0.370520,1.288156,1.144554,1.518917,1.193280,121.459940,195.811366,"Hidden Size=[15], regularizer=0.05, learning_r..."
3,model_24_3_0,0.184056,0.139581,-3.344434,0.488774,0.011959,1.336116,1.408943,1.034032,0.648860,0.841446,1.023375,1.155905,1.343555,1.205114,161.420466,260.149408,"Hidden Size=[20], regularizer=0.05, learning_r..."
4,model_44_8_0,0.180631,0.223218,-3.845089,0.160952,-0.388192,1.341725,1.271987,1.454913,1.239924,1.347418,1.698303,1.158328,1.255388,1.207641,201.412088,324.518547,"Hidden Size=[25], regularizer=0.05, learning_r..."
6,model_16_6_0,0.164717,0.068968,-0.409541,0.457861,-0.109082,1.367783,1.524574,0.463661,0.093154,0.278408,0.925457,1.169523,1.409118,1.219312,145.373618,234.351553,"Hidden Size=[18], regularizer=0.05, learning_r..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
289,model_40_0_0,-0.148507,-0.098410,-0.600830,-11.443534,-0.505645,1.880691,1.798657,2.429670,0.479855,1.454763,1.324314,1.371383,1.377591,1.429765,192.736721,310.967676,"Hidden Size=[24], regularizer=0.05, learning_r..."
290,model_36_9_0,-0.148539,-0.135068,-0.361192,-0.404110,-0.316853,1.880743,1.858684,8.550980,2.009794,5.280387,1.671820,1.371402,1.399492,1.429785,184.736666,298.092118,"Hidden Size=[23], regularizer=0.05, learning_r..."
291,model_3_7_0,-0.150115,-0.085290,-0.226554,-1.162677,-0.117914,1.883324,1.777173,7.729090,0.328421,4.028755,1.434881,1.372343,1.746021,1.430766,120.733923,195.085348,"Hidden Size=[15], regularizer=0.05, learning_r..."
292,model_43_4_0,-0.151111,-0.118951,-0.839887,0.104345,-0.415016,1.884956,1.832293,11.454773,0.253656,5.854215,0.735707,1.372937,1.358788,1.431386,200.732192,323.838650,"Hidden Size=[25], regularizer=0.05, learning_r..."
